# SQL — Analytical SQL: YoY Growth, Cohort Analysis, Funnel
---

<div style="padding:15px;border-left:8px solid #fa709a;background:#fff0f5;border-radius:4px;">
<strong>Core Insight:</strong> Analytical SQL separates analysts from engineers.
YoY, cohort, and funnel queries appear in every data engineering interview.
Master these three patterns and you can answer 80% of business intelligence questions
with SQL alone — no Python required.
</div>

### Why It Matters
At Citi, cohort analysis tracked server cohorts by provisioning quarter:
*"Of servers provisioned in Q1 2024, how many are still active in Q4 2025?"*
The same pattern used for user retention analysis works for infrastructure lifecycle.

## 🧠 Three Analytical Patterns

| Pattern | Business Question | SQL Technique |
|---------|-----------------|---------------|
| **YoY Growth** | "Revenue this month vs same month last year?" | `LAG(value, 12)` or self-join on `YEAR()` |
| **Cohort Analysis** | "Of users who joined in Jan, how many are still active 3 months later?" | `FIRST_VALUE()` + date arithmetic + GROUP BY cohort |
| **Funnel Analysis** | "How many users completed each step of checkout?" | Conditional aggregation with `COUNT(CASE WHEN...)` |

### The Window Function Toolkit
```sql
-- Running total
SUM(revenue) OVER (ORDER BY date)

-- Row-by-row comparison (YoY needs LAG of 12 months)
LAG(value, 1) OVER (PARTITION BY server_id ORDER BY month)

-- Rank within a group
RANK() OVER (PARTITION BY category ORDER BY value DESC)

-- First value in a group (cohort anchor)
FIRST_VALUE(date) OVER (PARTITION BY user_id ORDER BY date)
```

In [ ]:
-- Pattern 1: Year-over-Year Growth
-- Given: monthly revenue table with (year, month, revenue)
-- Question: Show each month with revenue, prior year revenue, and YoY growth %

-- Setup (simulate with values)
WITH monthly_revenue AS (
    SELECT 2024 AS yr, 1 AS mo, 1200000 AS revenue UNION ALL
    SELECT 2024, 2, 1350000 UNION ALL
    SELECT 2024, 3, 1280000 UNION ALL
    SELECT 2023, 1, 980000 UNION ALL
    SELECT 2023, 2, 1050000 UNION ALL
    SELECT 2023, 3, 1100000
),
-- Method 1: Self-join
yoy_self_join AS (
    SELECT
        curr.yr,
        curr.mo,
        curr.revenue AS curr_revenue,
        prev.revenue AS prev_revenue,
        ROUND((curr.revenue - prev.revenue) * 100.0 / prev.revenue, 1) AS yoy_pct
    FROM monthly_revenue curr
    LEFT JOIN monthly_revenue prev
        ON curr.yr = prev.yr + 1
        AND curr.mo = prev.mo
),
-- Method 2: LAG window function (cleaner)
yoy_lag AS (
    SELECT
        yr, mo, revenue,
        LAG(revenue, 12) OVER (ORDER BY yr, mo) AS prev_year_revenue
    FROM monthly_revenue
)
SELECT
    yr, mo, revenue,
    prev_year_revenue,
    ROUND((revenue - prev_year_revenue) * 100.0 / prev_year_revenue, 1) AS yoy_pct
FROM yoy_lag
WHERE prev_year_revenue IS NOT NULL
ORDER BY yr, mo;

In [ ]:
-- Pattern 2: Cohort Analysis
-- Given: server provisioning data (server_id, provisioned_date, decommissioned_date)
-- Question: For each provisioning cohort (quarter), what % are still active after N quarters?

-- Simulate server lifecycle data
WITH servers AS (
    SELECT 1 AS server_id, DATE('2024-01-15') AS provisioned, NULL AS decommissioned UNION ALL
    SELECT 2, DATE('2024-01-20'), DATE('2024-06-01') UNION ALL
    SELECT 3, DATE('2024-04-10'), NULL UNION ALL
    SELECT 4, DATE('2024-04-15'), DATE('2024-09-01') UNION ALL
    SELECT 5, DATE('2024-07-01'), NULL UNION ALL
    SELECT 6, DATE('2024-07-10'), NULL
),
-- Step 1: Assign each server to a cohort (provisioning quarter)
with_cohort AS (
    SELECT
        server_id,
        provisioned,
        decommissioned,
        -- Cohort = first quarter of provisioning year
        DATE(provisioned, 'start of month', '-' ||
            ((strftime('%m', provisioned) - 1) % 3) || ' months') AS cohort_start
    FROM servers
),
-- Step 2: Count cohort size
cohort_sizes AS (
    SELECT cohort_start, COUNT(*) AS cohort_size
    FROM with_cohort
    GROUP BY cohort_start
),
-- Step 3: Count still active at end of 2024 (no decommission date or decommissioned after)
still_active AS (
    SELECT cohort_start, COUNT(*) AS active_count
    FROM with_cohort
    WHERE decommissioned IS NULL OR decommissioned > DATE('2024-12-31')
    GROUP BY cohort_start
)
SELECT
    cs.cohort_start,
    cs.cohort_size,
    COALESCE(sa.active_count, 0) AS still_active,
    ROUND(COALESCE(sa.active_count, 0) * 100.0 / cs.cohort_size, 0) AS retention_pct
FROM cohort_sizes cs
LEFT JOIN still_active sa ON cs.cohort_start = sa.cohort_start
ORDER BY cs.cohort_start;

In [ ]:
-- Pattern 3: Funnel Analysis
-- Given: user events with (user_id, event_type, event_time)
-- Question: How many users completed each step of: view → add_to_cart → checkout → purchase?

-- Simulate checkout funnel events
WITH events AS (
    SELECT 1 AS user_id, 'view' AS event_type UNION ALL
    SELECT 1, 'add_to_cart' UNION ALL
    SELECT 1, 'checkout' UNION ALL
    SELECT 1, 'purchase' UNION ALL
    SELECT 2, 'view' UNION ALL
    SELECT 2, 'add_to_cart' UNION ALL
    SELECT 3, 'view' UNION ALL
    SELECT 4, 'view' UNION ALL
    SELECT 4, 'add_to_cart' UNION ALL
    SELECT 4, 'checkout'
),
-- Conditional aggregation: count distinct users who completed each step
funnel AS (
    SELECT
        COUNT(DISTINCT CASE WHEN event_type = 'view'        THEN user_id END) AS step1_view,
        COUNT(DISTINCT CASE WHEN event_type = 'add_to_cart' THEN user_id END) AS step2_cart,
        COUNT(DISTINCT CASE WHEN event_type = 'checkout'    THEN user_id END) AS step3_checkout,
        COUNT(DISTINCT CASE WHEN event_type = 'purchase'    THEN user_id END) AS step4_purchase
    FROM events
)
SELECT
    step1_view,
    step2_cart,
    step3_checkout,
    step4_purchase,
    -- Conversion rates between steps
    ROUND(step2_cart * 100.0 / step1_view, 0) AS view_to_cart_pct,
    ROUND(step3_checkout * 100.0 / step2_cart, 0) AS cart_to_checkout_pct,
    ROUND(step4_purchase * 100.0 / step3_checkout, 0) AS checkout_to_purchase_pct,
    ROUND(step4_purchase * 100.0 / step1_view, 0) AS overall_conversion_pct
FROM funnel;

## 🎤 5 Interview Q&A

**Q1: What is the difference between LAG and LEAD?**
`LAG(col, n)` returns the value from n rows BEFORE the current row (look backward).
`LEAD(col, n)` returns the value from n rows AFTER the current row (look forward).
Both require `OVER (ORDER BY ...)`. Use LAG for YoY comparisons (prior period), use LEAD
for "time until next event" calculations.

---

**Q2: How would you find users who completed step A but never step B?**
Use `COUNT(CASE WHEN event = 'A' THEN 1 END) > 0` in a HAVING clause,
combined with `COUNT(CASE WHEN event = 'B' THEN 1 END) = 0`:
```sql
SELECT user_id FROM events
GROUP BY user_id
HAVING SUM(CASE WHEN event = 'view' THEN 1 ELSE 0 END) > 0
   AND SUM(CASE WHEN event = 'purchase' THEN 1 ELSE 0 END) = 0;
```

---

**Q3: What is the difference between RANK(), DENSE_RANK(), and ROW_NUMBER()?**
All three number rows within a partition. With ties:
- `ROW_NUMBER()` assigns unique sequential numbers (1,2,3,4 — no ties)
- `RANK()` skips numbers after ties (1,2,2,4 — skips 3)
- `DENSE_RANK()` does not skip (1,2,2,3 — compact)
Use RANK for competitions (ties share place but the next rank skips). Use DENSE_RANK for top-N grouping.

---

**Q4: How do you calculate a 7-day rolling average in SQL?**
```sql
AVG(revenue) OVER (
    ORDER BY date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
)
```
`ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` defines a 7-row window (6 before + current).
Use `RANGE BETWEEN INTERVAL '6' DAY PRECEDING AND CURRENT ROW` for date-based ranges with gaps.

---

**Q5: What is the Citi application of cohort analysis?**
Citi capacity team used cohort analysis on server provisioning data instead of user events.
Cohort = provisioning quarter. Retention = still active (not decommissioned) after N quarters.
The analysis revealed 340 servers provisioned but not decommissioned — a "shadow fleet" costing
in licensing and maintenance. Identifying the cohort that had the worst decommissioning compliance
let the team target the right infrastructure owners for cleanup.

## 📚 Key Terms

| Term | Definition |
|------|------------|
| **Window Function** | Aggregate function that operates over a sliding window of rows without collapsing the result set |
| **PARTITION BY** | Divides rows into groups for window functions — like GROUP BY but keeps all rows |
| **ORDER BY (in window)** | Defines row order within each partition for LAG, LEAD, RANK, running totals |
| **LAG(col, n)** | Value from n rows before current row — used for period-over-period comparisons |
| **LEAD(col, n)** | Value from n rows after current row — used for time-to-next-event |
| **ROWS BETWEEN** | Explicit window frame — `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` = 7-row window |
| **Cohort** | A group defined by a shared characteristic at a point in time — "users who joined in January" |
| **Funnel** | A sequence of steps where users drop off at each stage — conversion rate per step |
| **YoY** | Year-over-Year — comparing the same period across different years |
| **Conditional Aggregation** | `SUM(CASE WHEN condition THEN value END)` — pivot-style aggregation without actual PIVOT |

## 💼 The Citi Narrative

**Context:** Citi capacity planning team — server fleet of 6,000 endpoints.
Infrastructure spans 12 teams, servers provisioned across 8 quarters.

**The Problem:** No visibility into "which servers should have been decommissioned but weren't."
Licensing costs were based on total server count, including long-inactive servers.

**The Cohort Analysis:** Applied the user retention pattern to server lifecycle:
- Cohort = provisioning quarter (Q1 2024, Q2 2024, etc.)
- "Retention" = still active (not decommissioned) after N quarters
- Expected: most servers decommissioned within 4-6 quarters as applications were retired

**The Finding:** Q1 2023 cohort had 340 servers still "active" after 8 quarters.
Normal decommission rate for this vintage was 85%. These 340 were a shadow fleet —
nobody had officially decommissioned them, but they weren't serving production traffic.

**Impact:** Identified $X in licensing cost reduction. Changed decommissioning process to
require a quarterly cohort report. Ownership of old servers now auto-escalated after 6 quarters.

**Interview Line:** *"The pattern came from user retention analysis, but the data was servers.
Cohort = provisioning quarter. Retention = still active. We found 340 servers that should
have been retired — using exactly the same SQL you'd write for user churn."*

In [ ]:
-- Practice: Write these analytical SQL queries

-- 1. Calculate month-over-month change in server count per environment
-- Table: dim_server(server_id, env, provisioned_date)
-- Expected output: env, month, server_count, prev_month_count, mom_change

-- 2. Find the top 3 servers by CPU for EACH environment
-- Table: fact_monitoring(server_id, env, cpu_pct)
-- Use RANK() or ROW_NUMBER() with PARTITION BY env

-- 3. Calculate 7-day rolling average CPU for server_id = 42
-- Table: daily_avg(server_id, date, avg_cpu)

-- Solutions:

-- 1. MoM change
print("Query 1 — MoM change:")
print("""
WITH monthly_count AS (
    SELECT
        env,
        DATE_TRUNC('month', provisioned_date) AS month,
        COUNT(*) AS server_count
    FROM dim_server
    GROUP BY env, DATE_TRUNC('month', provisioned_date)
)
SELECT
    env, month, server_count,
    LAG(server_count) OVER (PARTITION BY env ORDER BY month) AS prev_count,
    server_count - LAG(server_count) OVER (PARTITION BY env ORDER BY month) AS mom_change
FROM monthly_count
ORDER BY env, month;
""")

-- 2. Top 3 per env
print("Query 2 — Top 3 per environment:")
print("""
SELECT * FROM (
    SELECT
        server_id, env, cpu_pct,
        RANK() OVER (PARTITION BY env ORDER BY cpu_pct DESC) AS rank
    FROM fact_monitoring
) ranked
WHERE rank <= 3;
""")

## 🎯 Summary

### The Three Patterns
| Pattern | Use case | Key SQL |
|---------|----------|---------|
| **YoY Growth** | Period comparisons | `LAG(value, 12)` or self-join |
| **Cohort Analysis** | Retention / lifecycle | `FIRST_VALUE()` + `GROUP BY cohort` |
| **Funnel** | Conversion rates | `COUNT(CASE WHEN step = X THEN 1 END)` |

### Interview Confidence Checklist
- [ ] Can write YoY using both LAG and self-join (and explain tradeoffs)
- [ ] Can build a cohort table from raw event/provisioning data
- [ ] Can calculate funnel conversion rates with conditional aggregation
- [ ] Can explain RANK vs DENSE_RANK vs ROW_NUMBER
- [ ] Can name the Citi narrative: 340 ghost servers found via cohort analysis

**"Simplicity and clarity is Gold."** — Sean's Study Mantra 🚀